In [ ]:
# Databricks notebook sourceMAGIC %md# 1) Create a bridge / concordance table

In [ ]:
spark.sql("CREATE DATABASE IF NOT EXISTS bupa_reference")spark.sql("""CREATE TABLE IF NOT EXISTS bupa_reference.map_provider_id (  source_provider_id STRING,   -- as in CLAIMS  target_provider_id STRING,   -- as in PROVIDERS master  status STRING,               -- 'unmapped','active','deprecated'  from_system STRING,  to_system STRING,  effective_date DATE) USING DELTA""")

%md# 2) Seed it with all claim-side provider codes (for stewardship)

In [ ]:
from pyspark.sql import functions as Fdef canon(c): return F.upper(F.regexp_replace(F.trim(F.col(c)), r"[^A-Z0-9_]", ""))claims_dist = (spark.table("bupa_silver.claims")    .filter(F.col("Provider_ID").isNotNull())    .select(canon("Provider_ID").alias("source_provider_id")).distinct()    .withColumn("target_provider_id", F.lit(None).cast("string"))    .withColumn("status", F.lit("unmapped"))    .withColumn("from_system", F.lit("CLAIMS"))    .withColumn("to_system", F.lit("MASTER"))    .withColumn("effective_date", F.current_date()))claims_dist.createOrReplaceTempView("v_new_claim_providers")spark.sql("""MERGE INTO bupa_reference.map_provider_id tUSING v_new_claim_providers sON t.source_provider_id = s.source_provider_idWHEN NOT MATCHED THEN INSERT *""")